In [0]:
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceDoesNotExist
import pyspark.pandas as ps
from typing import Dict, List

In [0]:
dbutils.widgets.text(
  "from_date", "date_sub(current_date(), 365)", "Start date (yyyy-MM-dd)"
  )

dbutils.widgets.text(
  "target_catalog", "", "Target Catalog"
  )
dbutils.widgets.text(
  "target_schema", "", "Target Schema"
  )
dbutils.widgets.text(
  "target_table", "", "Target Table"
  )

In [0]:
assert dbutils.widgets.get("target_catalog") != "", "target_catalog should not be empty"
assert dbutils.widgets.get("target_schema") != "", "target_schema should not be empty"
assert dbutils.widgets.get("target_table") != "", "target_table should not be empty"

In [0]:
def get_errors_for_task(run_id: int, w: WorkspaceClient) -> Dict[str, str]:
  task_run_output = w.jobs.get_run_output(run_id = run_id).as_dict()
  assert task_run_output.get("error") is not None, "Job run output has no errors"
  return {
    "error": task_run_output.get("error"),
    "error_trace": task_run_output.get("error_trace")
  }


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(CONCAT(:target_catalog, ".", :target_schema));

CREATE TABLE IF NOT EXISTS IDENTIFIER(
  CONCAT(:target_catalog, ".", :target_schema, ".", :target_table)
) (
  run_id string,
  error string,
  error_trace string,
  update_time timestamp,
  error_classification string
);

In [0]:
job_task_run_failures = spark.sql(
    f"""
select
  run_id, result_state, termination_code -- this refers to task_run_id
From
  system.lakeflow.job_task_run_timeline
WHERE
  period_start_time >= {dbutils.widgets.get("from_date")}
  AND result_state IN ("ERROR", "FAILURE")
"""
)

In [0]:
run_ids = [x.run_id for x in job_task_run_failures.select("run_id").collect()]

In [0]:
w = WorkspaceClient()

In [0]:
out = {"run_id": [], "error": [], "error_trace": []}
for run_id in run_ids:
    try:
        errors = get_errors_for_task(run_id=run_id, w=w)
        out["run_id"] += [run_id]
        out["error"] += [errors.get("error")]
        out["error_trace"] += [errors.get("error_trace")]
    except ResourceDoesNotExist as e:
        pass # don't do anything
    except Exception as e:
        pass # don't do anything

In [0]:
psf = ps.DataFrame.from_dict(out)
pdf = psf.to_spark().withColumn("update_time", F.current_timestamp()).withColumn("error_classification", F.lit(None).cast("string"))

In [0]:
# We write this out so we can use it later for incremental AI classification
pdf.write.mode('overwrite').saveAsTable(f"{dbutils.widgets.get('target_catalog')}.{dbutils.widgets.get('target_schema')}.{dbutils.widgets.get('target_table')}_holding")

In [0]:
from delta.tables import *

target_table = DeltaTable.forName(spark, f"{dbutils.widgets.get('target_catalog')}.{dbutils.widgets.get('target_schema')}.{dbutils.widgets.get('target_table')}")
(target_table.alias("target").merge(pdf.alias('updates'), "updates.run_id = target.run_id")
  .whenMatchedUpdate(
    set = {
        "error": "updates.error",
        "error_trace": "updates.error_trace",
        "update_time": "updates.update_time"
    }
  )
  .whenNotMatchedInsertAll()
  .execute()
)